In [5]:
import polars as pl
import pandas as pd
from pathlib import Path

In [2]:
print("pandas:", pd.__version__)
print("polars:", pl.__version__)

pandas: 3.0.2
polars: 1.39.3


In [6]:
data_dir = Path("data_polars_handson")
data_dir.mkdir(exist_ok=True)

In [3]:
raw = [
    {"order_id": 1001, "customer": "Alice",   "region": "East",  "category": "Stock", "quantity": 10, "unit_price": 120.5, "discount": 0.05, "order_date": "2026-03-01", "is_member": True,  "note": "first"},
    {"order_id": 1002, "customer": "Bob",     "region": "West",  "category": "Bond",  "quantity":  5, "unit_price":  80.0, "discount": 0.00, "order_date": "2026-03-02", "is_member": False, "note": None},
    {"order_id": 1003, "customer": "Charlie", "region": "East",  "category": "Fund",  "quantity":  8, "unit_price": 150.0, "discount": 0.10, "order_date": "2026-03-02", "is_member": True,  "note": "vip"},
    {"order_id": 1004, "customer": "Diana",   "region": "North", "category": "Stock", "quantity": 12, "unit_price": 110.0, "discount": 0.03, "order_date": "2026-03-03", "is_member": True,  "note": ""},
    {"order_id": 1005, "customer": "Evan",    "region": "South", "category": "Fund",  "quantity":  3, "unit_price": 200.0, "discount": 0.15, "order_date": "2026-03-03", "is_member": False, "note": None},
    {"order_id": 1006, "customer": "Fiona",   "region": "West",  "category": "Stock", "quantity":  7, "unit_price": 130.0, "discount": 0.00, "order_date": "2026-03-04", "is_member": True,  "note": "campaign"},
    {"order_id": 1007, "customer": "George",  "region": "East",  "category": "Bond",  "quantity": 15, "unit_price":  75.5, "discount": 0.08, "order_date": "2026-03-04", "is_member": False, "note": None},
    {"order_id": 1008, "customer": "Hana",    "region": "South", "category": "Stock", "quantity":  6, "unit_price": 140.0, "discount": 0.05, "order_date": "2026-03-05", "is_member": True,  "note": "repeat"},
    {"order_id": 1009, "customer": "Ivan",    "region": "North", "category": "Bond",  "quantity":  9, "unit_price":  90.0, "discount": 0.02, "order_date": "2026-03-05", "is_member": False, "note": None},
    {"order_id": 1010, "customer": "Julia",   "region": "West",  "category": "Fund",  "quantity":  4, "unit_price": 210.0, "discount": 0.12, "order_date": "2026-03-06", "is_member": True,  "note": "vip"},
]

In [7]:
pdf_raw = pd.DataFrame(raw)
csv_path = data_dir / "sales.csv"
pdf_raw.to_csv(csv_path, index=False)
print(csv_path)
display(pdf_raw.head())

data_polars_handson/sales.csv


,order_id,customer,region,category,quantity,unit_price,discount,order_date,is_member,note
0,1001,Alice,East,Stock,10,120.5,0.05,2026-03-01,True,first
1,1002,Bob,West,Bond,5,80.0,0.00,2026-03-02,False,NaN
2,1003,Charlie,East,Fund,8,150.0,0.10,2026-03-02,True,vip
3,1004,Diana,North,Stock,12,110.0,0.03,2026-03-03,True,
4,1005,Evan,South,Fund,3,200.0,0.15,2026-03-03,False,NaN


In [8]:
# pandas
pdf = pd.read_csv(csv_path, parse_dates=["order_date"])
display(pdf.head())
print(pdf.dtypes)
print("index:", type(pdf.index))

,order_id,customer,region,category,quantity,unit_price,discount,order_date,is_member,note
0,1001,Alice,East,Stock,10,120.5,0.05,2026-03-01,True,first
1,1002,Bob,West,Bond,5,80.0,0.00,2026-03-02,False,NaN
2,1003,Charlie,East,Fund,8,150.0,0.10,2026-03-02,True,vip
3,1004,Diana,North,Stock,12,110.0,0.03,2026-03-03,True,NaN
4,1005,Evan,South,Fund,3,200.0,0.15,2026-03-03,False,NaN


order_id               int64
customer                 str
region                   str
category                 str
quantity               int64
unit_price           float64
discount             float64
order_date    datetime64[us]
is_member               bool
note                     str
dtype: object
index: <class 'pandas.RangeIndex'>


In [9]:
# Polars
pldf = pl.read_csv(
    csv_path,
    try_parse_dates=True,
    schema_overrides={"is_member": pl.Boolean},
)

In [11]:
print(pldf.head())

shape: (5, 10)
┌──────────┬──────────┬────────┬──────────┬───┬──────────┬────────────┬───────────┬───────┐
│ order_id ┆ customer ┆ region ┆ category ┆ … ┆ discount ┆ order_date ┆ is_member ┆ note  │
│ ---      ┆ ---      ┆ ---    ┆ ---      ┆   ┆ ---      ┆ ---        ┆ ---       ┆ ---   │
│ i64      ┆ str      ┆ str    ┆ str      ┆   ┆ f64      ┆ date       ┆ bool      ┆ str   │
╞══════════╪══════════╪════════╪══════════╪═══╪══════════╪════════════╪═══════════╪═══════╡
│ 1001     ┆ Alice    ┆ East   ┆ Stock    ┆ … ┆ 0.05     ┆ 2026-03-01 ┆ true      ┆ first │
│ 1002     ┆ Bob      ┆ West   ┆ Bond     ┆ … ┆ 0.0      ┆ 2026-03-02 ┆ false     ┆ null  │
│ 1003     ┆ Charlie  ┆ East   ┆ Fund     ┆ … ┆ 0.1      ┆ 2026-03-02 ┆ true      ┆ vip   │
│ 1004     ┆ Diana    ┆ North  ┆ Stock    ┆ … ┆ 0.03     ┆ 2026-03-03 ┆ true      ┆ null  │
│ 1005     ┆ Evan     ┆ South  ┆ Fund     ┆ … ┆ 0.15     ┆ 2026-03-03 ┆ false     ┆ null  │
└──────────┴──────────┴────────┴──────────┴───┴──────────┴───────

In [12]:
print(pldf.schema)

Schema({'order_id': Int64, 'customer': String, 'region': String, 'category': String, 'quantity': Int64, 'unit_price': Float64, 'discount': Float64, 'order_date': Date, 'is_member': Boolean, 'note': String})
